# RCT Design & Difference-in-Differences Estimation

**Author:** Iván de Luna  
**Topics:** Experimental Design · Statistical Power · Causal Inference

---

## Overview

This notebook walks through the two core stages of running a **Randomized Controlled Trial (RCT)**:

1. **Designing the experiment** — using statistical power analysis to determine the minimum sample size needed to reliably detect an effect.
2. **Analyzing the results** — using the **Difference-in-Differences (DiD)** estimator to measure the causal impact of a treatment.

These methods are foundational in economics, public policy evaluation, A/B testing, and clinical research. The two parts of the notebook are intentionally sequential: you run a power analysis *before* collecting data, then apply DiD *after*.

---

### Contents
- [Part 1: Statistical Power Analysis](#part1)
- [Part 2: Difference-in-Differences Estimation](#part2)
- [Summary & Takeaways](#summary)

### Requirements
```
pandas, numpy, matplotlib, scipy, statsmodels
```

In [ ]:
# --- Dependencies ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import norm
import statsmodels.stats.power as smp
import statsmodels.formula.api as smf

# Consistent style across all plots
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

np.random.seed(42)

<a id='part1'></a>
## Part 1 — Statistical Power Analysis

### What is Statistical Power?

Before running any experiment, we need to know: *how many observations do we need to trust the results?*

**Statistical power** is the probability that a test correctly rejects the null hypothesis when the alternative is true — in other words, the probability of detecting a real effect when one exists. It is defined as:

$$\text{Power} = 1 - \beta$$

where $\beta$ is the **Type II error rate** (the probability of a false negative). By convention, studies aim for **power ≥ 0.80**.

Power depends on three factors:
| Factor | Symbol | Effect on Power |
|---|---|---|
| Effect size | $d$ (Cohen's *d*) | Larger effect → higher power |
| Significance level | $\alpha$ | Higher $\alpha$ → higher power (but more false positives) |
| Sample size | $n$ | Larger sample → higher power |

**Cohen's *d*** standardizes the effect size as the difference in means divided by the pooled standard deviation:

$$d = \frac{\mu_1 - \mu_2}{\sigma}$$

Conventional benchmarks: small = 0.2, medium = 0.5, large = 0.8.

In [ ]:
# --- Sample Size Calculation ---
# We define the study parameters upfront, as would be done in a pre-registration

EFFECT_SIZE = 0.5  # Medium effect (Cohen's d)
ALPHA       = 0.05  # 5% significance level (two-tailed)
TARGET_POWER = 0.80  # 80% power — standard threshold

analysis = smp.TTestIndPower()
n_per_group = analysis.solve_power(
    effect_size=EFFECT_SIZE,
    alpha=ALPHA,
    power=TARGET_POWER
)

print(f"Study parameters:")
print(f"  Effect size (Cohen's d) : {EFFECT_SIZE}")
print(f"  Significance level (α)  : {ALPHA}")
print(f"  Target power (1 - β)    : {TARGET_POWER}")
print(f"")
print(f"→ Minimum sample size per group : {round(n_per_group)} observations")
print(f"→ Total sample size             : {round(n_per_group) * 2} observations")

With the study parameters above, we need at least **64 observations per group** (128 total) to have an 80% chance of detecting a medium-sized effect at a 5% significance level.

But what happens if the true effect is smaller or larger than assumed? The next plot shows how power varies across a range of effect sizes, holding sample size constant.

In [ ]:
# --- Power vs. Effect Size Curve ---
effect_sizes = np.arange(0.1, 1.05, 0.05)
powers = [
    analysis.solve_power(effect_size=es, alpha=ALPHA, nobs1=n_per_group)
    for es in effect_sizes
]

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(effect_sizes, powers, color='steelblue', linewidth=2)
ax.axhline(TARGET_POWER, color='gray', linestyle='--', linewidth=1, label=f'Target power = {TARGET_POWER}')
ax.axvline(EFFECT_SIZE, color='tomato', linestyle='--', linewidth=1, label=f'Assumed effect size = {EFFECT_SIZE}')
ax.scatter([EFFECT_SIZE], [TARGET_POWER], color='tomato', zorder=5)

ax.fill_between(effect_sizes, powers, TARGET_POWER,
                where=[p < TARGET_POWER for p in powers],
                alpha=0.15, color='tomato', label='Underpowered region')

ax.set_xlabel('Effect Size (Cohen\'s d)')
ax.set_ylabel('Statistical Power')
ax.set_title(f'Power vs. Effect Size  (n = {round(n_per_group)} per group, α = {ALPHA})')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The shaded red region marks the **underpowered zone**: if the true effect is smaller than assumed (d < 0.5) and we collect only 64 observations per group, we are more likely to miss it than to detect it — a critical risk in experiment design.

---

### Visualizing Power as Overlapping Distributions

Power can be understood geometrically. Consider two distributions:
- **H₀** — the null hypothesis: there is no effect (mean = 0)
- **H₁** — the alternative hypothesis: there is an effect (mean = d)

We reject H₀ when the test statistic falls beyond the critical value $z_{α/2}$. The **orange area** under H₁ beyond that threshold is the power — the probability of correctly rejecting H₀ when H₁ is true. The **blue areas** under H₀ represent the Type I error rate $\alpha$.

In [ ]:
# --- Two-Distribution Power Visualization ---
mu0   = 0            # Null distribution mean
mu1   = EFFECT_SIZE  # Alternative distribution mean (Cohen's d with σ=1)
sigma = 1
critical_value = norm.ppf(1 - ALPHA / 2)

x  = np.linspace(-3.5, 3.5, 600)
y0 = norm.pdf(x, mu0, sigma)
y1 = norm.pdf(x, mu1, sigma)

power_value = 1 - norm.cdf(critical_value, mu1, sigma)
type2_error = 1 - power_value  # β

fig, ax = plt.subplots(figsize=(10, 5))

# Distribution curves
ax.plot(x, y0, color='steelblue', linewidth=2, label='H₀ (no effect)')
ax.plot(x, y1, color='darkorange', linewidth=2, label='H₁ (effect exists)')

# Type I error (α) — rejection regions under H₀
ax.fill_between(x, y0, where=(x > critical_value),  color='steelblue', alpha=0.4, label=f'Type I error (α/2)')
ax.fill_between(x, y0, where=(x < -critical_value), color='steelblue', alpha=0.4)

# Power — rejection region under H₁
ax.fill_between(x, y1, where=(x > critical_value), color='darkorange', alpha=0.5, label=f'Power = {power_value:.2f}')

# Type II error (β) — missed detections under H₁
ax.fill_between(x, y1, where=(x <= critical_value), color='green', alpha=0.15, label=f'Type II error β = {type2_error:.2f}')

# Critical value lines
ax.axvline( critical_value, color='red', linestyle='--', linewidth=1)
ax.axvline(-critical_value, color='red', linestyle='--', linewidth=1, label=f'Critical values (±{critical_value:.2f})')

ax.set_xlabel('Test Statistic (z-score)')
ax.set_ylabel('Probability Density')
ax.set_title('Statistical Power — H₀ vs. H₁ Distributions')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Effect size (d)  : {EFFECT_SIZE}")
print(f"Critical value   : ±{critical_value:.3f}")
print(f"Power (1 - β)    : {power_value:.2f}  → {power_value*100:.1f}% chance of detecting the effect")
print(f"Type II error (β): {type2_error:.2f}  → {type2_error*100:.1f}% chance of missing it")

<a id='part2'></a>
## Part 2 — Difference-in-Differences Estimation

### What is Difference-in-Differences?

Once the experiment is run and data collected, we estimate the **causal effect of the treatment** using the DiD estimator.

DiD compares the change over time in the **treatment group** against the change over time in the **control group**. The key insight is that by differencing out the time trend, we isolate the effect of the treatment:

$$\hat{\delta}_{DiD} = \underbrace{(\bar{Y}_{treatment,post} - \bar{Y}_{treatment,pre})}_{\text{treatment group change}} - \underbrace{(\bar{Y}_{control,post} - \bar{Y}_{control,pre})}_{\text{control group change}}$$

The key identifying assumption is the **parallel trends assumption**: in the absence of treatment, both groups would have followed the same time trend.

### Two equivalent implementations

The DiD estimate can be computed:
1. **Manually** — by calculating group means and taking differences.
2. **Via regression** — by regressing the outcome on a treatment dummy, a time dummy, and their interaction. The coefficient on the interaction term is the DiD estimate. This approach is more powerful in practice because it allows for covariates and standard error computation.

In [ ]:
# --- Simulate RCT Data ---
# We create a balanced panel: 100 units observed in pre and post periods
# The treatment group receives an intervention with a true effect of +3 units

N_UNITS      = 100
TRUE_EFFECT  = 3.0   # The ground truth we want to recover

data = pd.DataFrame({
    'unit' : np.arange(N_UNITS),
    'group': np.random.choice(['control', 'treatment'], size=N_UNITS),
    'time' : np.random.choice(['pre', 'post'], size=N_UNITS),
    'y'    : np.random.normal(loc=10, scale=2, size=N_UNITS)
})

# Apply treatment effect only to treatment group in the post period
treated_post = (data['group'] == 'treatment') & (data['time'] == 'post')
data.loc[treated_post, 'y'] += TRUE_EFFECT

# Create binary dummies for regression
data['treated']  = (data['group'] == 'treatment').astype(int)  # 1 = treatment group
data['post']     = (data['time']  == 'post').astype(int)       # 1 = post period

print("Dataset shape:", data.shape)
print("\nGroup counts:")
print(data.groupby(['group', 'time']).size().unstack())

In [ ]:
# --- Method 1: Manual DiD (Group Means) ---
means = data.groupby(['group', 'time'])['y'].mean().unstack()

diff_control   = means.loc['control',   'post'] - means.loc['control',   'pre']
diff_treatment = means.loc['treatment', 'post'] - means.loc['treatment', 'pre']
did_manual     = diff_treatment - diff_control

print("Group means (outcome by group and period):")
print(means.round(3))
print()
print(f"Change in control group   (post - pre): {diff_control:.3f}")
print(f"Change in treatment group (post - pre): {diff_treatment:.3f}")
print()
print(f"DiD Estimate (manual): {did_manual:.3f}")
print(f"True effect           : {TRUE_EFFECT:.3f}")

In [ ]:
# --- Method 2: Regression-based DiD ---
# The model is: y = β₀ + β₁·treated + β₂·post + δ·(treated × post) + ε
# δ is the DiD estimator — the coefficient on the interaction term

model  = smf.ols('y ~ treated + post + treated:post', data=data).fit()

did_regression = model.params['treated:post']
did_ci         = model.conf_int().loc['treated:post']
did_pvalue     = model.pvalues['treated:post']

print("Regression-based DiD Results")
print("=" * 40)
print(f"DiD Estimate (interaction coef): {did_regression:.3f}")
print(f"95% Confidence Interval        : [{did_ci[0]:.3f}, {did_ci[1]:.3f}]")
print(f"p-value                        : {did_pvalue:.4f}")
print(f"True effect                    : {TRUE_EFFECT:.3f}")
print()
print("Full regression summary:")
print(model.summary().tables[1])

In [ ]:
# --- DiD Visualization ---
# A classic "two lines" plot showing parallel pre-trends and treatment divergence

means_plot = data.groupby(['group', 'time'])['y'].mean().reset_index()
time_order = {'pre': 0, 'post': 1}
means_plot['time_num'] = means_plot['time'].map(time_order)

ctrl = means_plot[means_plot['group'] == 'control'].sort_values('time_num')
trt  = means_plot[means_plot['group'] == 'treatment'].sort_values('time_num')

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot([0, 1], ctrl['y'].values, 'o-', color='steelblue',  linewidth=2, markersize=8, label='Control')
ax.plot([0, 1], trt['y'].values,  's-', color='darkorange', linewidth=2, markersize=8, label='Treatment')

# Counterfactual: what treatment would look like without the intervention
counterfactual_post = trt['y'].values[0] + diff_control  # parallel trends assumption
ax.plot([0, 1], [trt['y'].values[0], counterfactual_post],
        '--', color='darkorange', linewidth=1.5, alpha=0.5, label='Counterfactual (parallel trends)')

# Annotate the DiD gap
ax.annotate('', xy=(1.0, trt['y'].values[1]),
            xytext=(1.0, counterfactual_post),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax.text(1.02, (trt['y'].values[1] + counterfactual_post) / 2,
        f'DiD = {did_manual:.2f}', fontsize=10, va='center')

ax.set_xticks([0, 1])
ax.set_xticklabels(['Pre-period', 'Post-period'])
ax.set_ylabel('Mean Outcome (y)')
ax.set_title('Difference-in-Differences — Group Means Over Time')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The dashed orange line represents the **counterfactual** — what would have happened to the treatment group if the intervention had not occurred, assuming the parallel trends assumption holds. The DiD estimate is the vertical gap between the actual post-period treatment mean and this counterfactual.

---

<a id='summary'></a>
## Summary & Key Takeaways

| Step | Method | Result |
|---|---|---|
| Pre-experiment design | Power analysis | 64 obs/group needed to detect d = 0.5 with 80% power |
| Post-experiment analysis (manual) | Mean differences | DiD ≈ 3.08 |
| Post-experiment analysis (regression) | OLS with interaction | DiD ≈ 3.08, p < 0.05 |

Both methods recover the true treatment effect of **+3 units** quite accurately. The small discrepancy is due to sampling noise — by construction, the simulated treatment effect is exactly 3.

**Key points to remember:**
- Always conduct a **power analysis before** collecting data. Under-powered studies waste resources and produce unreliable results.
- The **regression-based DiD** is preferred in practice: it yields standard errors, confidence intervals, and can be extended with control variables.
- The **parallel trends assumption** is the critical identifying assumption. In observational settings (not true RCTs), this should be tested with pre-treatment data.
- In a true RCT, randomization ensures that the parallel trends assumption holds by design.